# ALS reconstruction → KRG-alsDenoise — v3.4

This notebook reconstructs a raw ALS CT scan, makes independent even/odd stacks,
and trains/applies a denoiser. **No earlier notebook is required.** First-time ALS
and n2i installation instructions are included below. The Python 3.8 ALS kernel
runs the lightweight `krg_workflow` package; the separate n2i executable runs PyTorch.

v3.4 moves reusable helpers out of notebook cells. You edit paths and parameters,
then call short actions on an explicit `run` object. Existing schema-4 plans,
queues, training YAML, logs and checkpoints remain compatible. v3.3 is preserved.
Fresh 2.5D runs default to the NAFNet adaptation; 3D and the original U-Net remain
available. Training is disabled until you explicitly set `RUN_TRAINING=True`.
No training was performed to validate this refactor.


## Read this first: what the notebook does

**Noise2Inverse360** trains a denoising model without a clean reference image. For each slice, it compares two reconstructions made from **different measured angles**: even-indexed projections and odd-indexed projections. The model learns features that agree between the two views. A third reconstruction from **all** projections supplies the image that will be denoised after training. The two training views share flats and darks, so their noise is not perfectly independent; inspect the result for removed features and remaining artifacts.

This notebook performs **reconstruction and denoising**. The ALS `microct` code performs the reconstruction; `KRG-alsDenoise` supplies the Noise2Inverse model training and inference commands. Reconstruction settings, especially center of rotation, must match across the full, even, and odd stacks. The notebook saves one plan and applies it to all three.

| Term | Meaning here |
|---|---|
| Projection | A raw 2D X-ray image acquired at one rotation angle. |
| Flat / dark | Calibration images acquired without the sample / without X-ray illumination. |
| Sinogram row | One detector row followed through all angles; it reconstructs one CT slice. |
| Center of rotation | Detector-center offset in pixels: `native center - detector width / 2`. Zero is the midpoint; negative offsets are valid. A wrong value can produce doubled edges. |
| Full / even / odd stack | TIFF slices reconstructed from all / even-indexed / odd-indexed projections. |
| 2.5D model | A 2D network that reads several neighboring slices as channels. |
| 3D model | A 3D network that reads and predicts cubic patches of the volume. |
| HDF5 virtual view | A small HDF5 file pointing at selected projections in the original file; it does not duplicate the raw images. |

**Input required:** one 180° parallel-beam HDF5 scan containing `/exchange/data` in `[angle, row, column]` order, `/exchange/theta` in degrees, and `/exchange/data_white` and `/exchange/data_dark`. The ALS path also needs detector dimensions, pixel size, angle count, sample distance, energy, angular range, and other standard ALS metadata. Cell 3 checks image layout and ALS metadata; cell 4 rechecks metadata before reconstruction. A folder of only reconstructed TIFFs cannot supply the independent even/odd training views; obtain the raw projections or separately reconstructed views before using Noise2Inverse360.

The detector width must be even. Flats and darks must be nonempty `[frame, row, column]` arrays matching the projections. Angles must increase monotonically and span approximately 180°. Metadata values used below must be finite, nonempty 1D arrays, not HDF5 scalar datasets.

| Required ALS metadata path | Meaning used by this notebook |
|---|---|
| `/measurement/instrument/detector/dimension_x`, `dimension_y` | Column and row counts, matching `/exchange/data`. |
| `/measurement/instrument/detector/pixel_size` | First entry: detector pixel size in **mm**; must be positive. |
| `/process/acquisition/rotation/num_angles`, `range` | Projection count and angular span in **degrees**, consistent with the image data and theta. |
| `/measurement/instrument/camera_motor_stack/setup/camera_distance` | Optional for this no-phase workflow. If present as a finite second entry, it is reported in mm; otherwise distance is recorded as unknown. A private metadata adapter avoids upstream indexing entry 1 when it is absent. |
| `/measurement/instrument/monochromator/energy` | First entry in **eV**. Required by the reader even though this notebook does not perform phase retrieval. |

A finite negative energy value can pass the reader but is not a valid physical energy; it does not enable phase retrieval. Obtain missing or malformed metadata from the beamline rather than substituting guessed values. The pixel-size conversion used for reconstruction is mm → cm, giving attenuation values in **1/cm**.

## First-time NERSC setup

This guide starts with a NERSC account, access to the raw scan, and permission to
clone `remyferu/KRG-alsDenoise`. No earlier notebook or installed CT environment
is required. Commands marked **terminal** are Bash commands on NERSC, not Python
notebook cells and not commands for your Windows computer. A **kernel** is the
Python process that executes notebook cells; selecting it is separate from
selecting the compute node on which Jupyter runs.

### 1. Understand the two environments

| Environment | What runs there | Main software | How it is selected |
|---|---|---|---|
| `als832` | Notebook cells, projection processing, COR and reconstruction | ALS `microct` source, TomoPy, ASTRA, dxchange, NumPy, widgets | Select **ALS CT + N2I notebook** as the Jupyter kernel |
| `n2i` | Denoising training and inference | This repository, PyTorch with CUDA | Notebook launches its absolute `denoise` executable as a subprocess |

Create both environments once. Keep the notebook in the ALS kernel during the
whole workflow; you do not need to switch to the n2i kernel for training.
The source checkouts contain code, while the Conda prefixes contain Python and
installed packages. Cloning a repository alone does not install its dependencies.

This notebook reconstructs through ALS/TomoPy/ASTRA; no Tomocupy environment is
required. The Configure training cell runs `denoise prepare` to create a training YAML. That command
may print Tomocupy reconstruction examples inherited from the original CLI;
ignore those examples here because cells 17â€“19 already produce/check the paired
stacks using ALS.

If an ALS-provided `microCT` environment is already working, you may use it for
the kernel after verifying the imports below. The
[ALS project](https://github.com/als-computing/microct#readme) describes its
beamline-specific service. The private installation below is an alternative for
standard Perlmutter Jupyter sessions. Their filesystem mounts may differ.

### 2. Create directories and download the source â€” login-node terminal

Log in to NERSC through SSH, or use a login-node Jupyter terminal for lightweight
installation work. Use a compute allocation for expensive compilation or data
processing. These paths match the notebook's defaults. The Conda module follows
the [NERSC custom-environment procedure](https://docs.nersc.gov/development/languages/python/nersc-python/#option-2-custom-conda-environment):

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
mkdir -p "$CT_SOURCE_DIR" "$CT_SOFTWARE_DIR"
cd "$CT_SOURCE_DIR"
git clone https://github.com/als-computing/microct.git microct
git -C "$CT_SOURCE_DIR/microct" checkout e4116984003294af8dabba9d32db3fccb5ef2253
git clone https://github.com/remyferu/KRG-alsDenoise.git KRG-alsDenoise
```

The KRG repository is private: use your normal GitHub HTTPS authentication, or
your configured SSH key with `git@github.com:remyferu/KRG-alsDenoise.git` as the
clone URL. If a directory already exists, inspect it and reuse the checkout;
do not rerun `git clone` into it or replace it. The ALS commit above matches the
backend reviewed for this notebook.

### 3. Install the ALS reconstruction environment â€” login-node terminal

For a fresh private installation, create `als832` from the pinned ALS source's
environment file, then add the notebook's supporting packages:

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
conda env create --prefix "$CT_SOFTWARE_DIR/als832" \
  -f "$CT_SOURCE_DIR/microct/environment.yml"
conda activate "$CT_SOFTWARE_DIR/als832"
conda install -c conda-forge h5py tifffile pyyaml pillow numexpr ipywidgets ipykernel
python -c 'import tomopy, dxchange, astra, h5py, tifffile, yaml, PIL, ipywidgets; print("ALS imports OK")'
python -m pip install --no-deps -e "$CT_SOURCE_DIR/KRG-alsDenoise/workflow"
krg-workflow doctor
python -m ipykernel install --user --name als832-n2i \
  --display-name "ALS CT + N2I notebook"
```

The [pinned ALS environment](https://github.com/als-computing/microct/blob/e4116984003294af8dabba9d32db3fccb5ef2253/environment.yml)
uses Python 3.8 and older Matplotlib/Jupyter pins. This is a source-based setup
recipe, not a freshly tested NERSC package lock. If Conda cannot solve it, keep an
existing working ALS environment and resolve the solver error before proceeding;
do not install TomoPy into n2i as a substitute. `ipywidgets` is needed by the ALS
backend itself. Zoom displays do not require an interactive Matplotlib backend.

In JupyterLab choose **Kernel â†’ Change Kernel â†’ ALS CT + N2I notebook**. Reload
JupyterLab if the kernel is missing. Changes to an already running kernel's
environment require a kernel restart. The first code cell prints `sys.executable`;
for this installation it should end in `microct-n2i-envs/als832/bin/python`.
See the [NERSC custom-kernel procedure](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-use-a-conda-environment-as-a-python-kernel).

The `krg-ct-workflow` package lives in `workflow/` and requires no Torch. Its
scientific dependencies are supplied by the ALS environment, so use `--no-deps`
to preserve those compatible versions. If pip/setuptools is too old for editable
installation, install `setuptools>=61,<76` and `pip<25.1` inside this Python 3.8
environment, then retry. Restart the notebook kernel after installing/updating it.

### 4. Install the denoising environment â€” login-node terminal

For a new n2i prefix, install a CUDA-enabled PyTorch build and this package with
its dependencies. The example uses an official PyTorch 2.7.1 / CUDA 11.8 wheel;
the notebook requires PyTorch >= 2.0. Use a site-compatible build when necessary.
[PyTorch wheel choices](https://pytorch.org/get-started/previous-versions/#v271)
and [NERSC guidance](https://docs.nersc.gov/machinelearning/pytorch/) explain the
custom-environment option.

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
conda create --prefix "$CT_SOFTWARE_DIR/n2i" python=3.11 pip
conda activate "$CT_SOFTWARE_DIR/n2i"
python -m pip install 'numpy<2.0'
python -m pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu118
python -m pip install -e "$CT_SOURCE_DIR/KRG-alsDenoise"
python -m pip check
MPLBACKEND=Agg PYTHONNOUSERSITE=1 "$CT_SOFTWARE_DIR/n2i/bin/denoise" train -h
```

Help must contain `--architecture` and its `nafnet25d` choice. There is no need to
clone NAFNet separately or install BasicSR. For an **existing working n2i prefix**,
skip `conda create` and the dependency installations; activate that prefix and run
`python -m pip install --no-deps -e "$HOME/microct-n2i-source/KRG-alsDenoise"`,
then check imports and CLI help. `--no-deps` is only for an environment with the
dependencies already installed. The repository's original
`envs/legacy/denoise_environment.yml` is a machine-specific upstream export, not the
portable first-time recipe for this notebook.

### 5. Start Jupyter on a compute allocation

Open [NERSC JupyterHub](https://jupyter.nersc.gov/), start a **single-node GPU**
compute server and open
`microct-n2i-source/KRG-alsDenoise/notebooks/Noise2Inverse360_HPC_v3.4.ipynb`.
For the full-volume example, choose **Exclusive GPU Node**, or a Configurable job
with sufficient host RAM and time. NERSC documents four GPUs and full node memory
for the exclusive GPU configuration; a shared GPU server has one GPU and a
quarter of the node's host resources. [Allocation options](https://docs.nersc.gov/services/jupyter/reference/).

Select **ALS CT + N2I notebook** as the kernel. Activating Conda or starting an
allocation in another terminal does not relocate or change the running kernel.
The workflow uses one node: training uses all assigned GPUs, inference uses one.
In a terminal opened **inside this compute Jupyter session**, check:

```bash
hostname
echo "Slurm job: ${SLURM_JOB_ID:-unset}"
echo "GPU assignment: ${CUDA_VISIBLE_DEVICES:-unset}"
nvidia-smi
"$HOME/microct-n2i-envs/n2i/bin/python" -c 'import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available(); print("Visible GPUs:", torch.cuda.device_count())'
```

This checks CUDA availability; it does not train a model. Keep the scheduler's
`CUDA_VISIBLE_DEVICES` assignment. If it is unset, resolve the allocation/kernel
setup before training instead of guessing identifiers from `nvidia-smi`. If either
environment needs site CUDA modules, apply them through a kernel/helper wrapper
following [NERSC's helper instructions](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-customize-a-kernel-with-a-helper-shell-script).
The absolute executable path alone cannot repair a missing native runtime.

### 6. Stage the scan and edit cell 2

The notebook does not download or copy scan data. Put your raw HDF5 scan in a
directory visible from the compute kernel. On standard Perlmutter, an example is
`$PSCRATCH/raw_microct/sample.h5`. Use your own source filename, and verify that
the file meets the projection/flat/dark/metadata requirements in **Read this first**.
Reconstructed TIFFs alone are insufficient for the notebook's projection splitting.

| Cell 2 input | What to enter |
|---|---|
| `INPUT_ROOT` | Folder containing your raw `.h5`/`.hdf5` files; replace the example folder or use an explicit absolute `Path` |
| `SCAN_PATH` | Absolute raw scan path; confirm it in cell 3 |
| `RUN_LABEL` | A name for this scan's particular COR/settings/model combination; use a new label when those change |
| `ALS_MICROCT_DIR` | Checkout containing `backend/ALS_recon_helper.py`, normally `Path.home() / 'microct-n2i-source' / 'microct'` |
| `DENOISE_CMD` | `[str(Path.home() / 'microct-n2i-envs' / 'n2i' / 'bin' / 'denoise')]` |
| `OUTPUT_ROOT_OVERRIDE` | Leave `None` to use `$PSCRATCH`/`$SCRATCH`, or enter an absolute output scratch directory |

In Python, `Path.home()` expands your home directory; `Path('$HOME/...')` does not
expand Bash variables. `/alsdata` and `/alsuser/pscratch` are service-specific
paths, not assumed mounts on standard Perlmutter. Stage to a stable input path:
moving the raw file later breaks the saved plan and virtual projection views.

### 7. Know where results and code live

Keep code, environments and notebooks in home (subject to quota); use scratch for
large input copies, TIFFs and active model outputs. Standard output layout:

```text
$HOME/microct-n2i-source/microct/backend/ALS_recon_helper.py
$HOME/microct-n2i-source/KRG-alsDenoise/notebooks/Noise2Inverse360_HPC_v3.4.ipynb
$HOME/microct-n2i-envs/als832/         # notebook Python
$HOME/microct-n2i-envs/n2i/           # denoising Python
$PSCRATCH/noise2inverse360/<scan>/<RUN_LABEL>/
  run_plan.json, scan_metadata.json
  projection_views/even.h5, odd.h5
  <scan>_rec/, <scan>_rec_0/, <scan>_rec_1/
  <scan>_rec_config.yaml
  TrainOutput/results/reference_metrics.csv
  TrainOutput/{resume,best_val_model,best_lcl_model,best_edge_model}.pth
  <scan>_denoised_slices_nafnet25d/
  <scan>_denoised_volume_2.5d_nafnet25d/
  logs/, training_history/
$PSCRATCH/noise2inverse360/reconstruction_queue_v3.json
```

The selected output root can differ from this example. Full/even/odd float32
stacks for 624 Ã— 2560 Ã— 2560 occupy about 46 GiB on disk. Training with four
processes holds about 122 GiB of paired input arrays in host RAM before overhead.
GPU VRAM is separate from host RAM. Read the volume section for its additional
memory needs. Scratch is working storage; preserve accepted data and small run
records in your normal durable storage before cleanup.

### 8. Update an existing installation

After stopping jobs that use this checkout, update it in a terminal:

```bash
cd "$HOME/microct-n2i-source/KRG-alsDenoise"
git status --short
git pull --ff-only
```

For the first upgrade from v3.3, also activate the ALS environment and install
`python -m pip install --no-deps -e "$HOME/microct-n2i-source/KRG-alsDenoise/workflow"`.
Open the new v3.4 notebook; the v3.3 notebook remains available.

Preserve your own notebook edits before pulling. Editable installation reads code
from this checkout, so routine code updates need no package reinstall; changed
dependencies or entry points do. Reopen the updated notebook and restart its kernel and rerun initialization. Do not update the pinned ALS checkout during an active run: changing
backend files invalidates its saved source identity.

### Setup troubleshooting

| Symptom | Next step |
|---|---|
| `ModuleNotFoundError: tomopy`, `astra` or `dxchange` in a cell | Check the printed kernel Python; select the ALS kernel and verify ALS imports |
| `ModuleNotFoundError: torch` from `denoise` | Check `DENOISE_CMD` and the n2i environment; PyTorch belongs there |
| CLI help lacks `--architecture` | Update/install KRG-alsDenoise in n2i; an older Noise2Inverse executable is being used |
| Invalid `matplotlib_inline` backend in the CLI | Confirm the subprocess uses `MPLBACKEND=Agg`; the notebook's launcher sets it |
| NumExpr thread-count error | Restart the kernel and run cell 1 first. If NumExpr is preloaded, set `NUMEXPR_MAX_THREADS` in the kernel launcher to at least the host's detected logical CPU count before Python starts |
| Kernel remains missing | Reload JupyterLab after registration; check the kernelspec points to an existing ALS Python |
| No accessible scratch or scan folder | Set cell 2's input and scratch paths to locations visible on the actual compute node |

If NumExpr loads before cell 1, inspect the host CPU count with
`getconf _NPROCESSORS_ONLN` in the compute-session terminal. In JupyterLab's text
editor open `$HOME/.local/share/jupyter/kernels/als832-n2i/kernel.json` and add an
`env` mapping at the same level as `argv` (keep the existing interpreter path):

```json
"env": {
  "NUMEXPR_MAX_THREADS": "256",
  "NUMEXPR_NUM_THREADS": "8"
}
```

`256` is an example: use a maximum at least as large as that host's detected
logical CPU count. Keep valid JSON commas between the existing fields, then
restart the kernel. Cell 1 reduces the initial active count to the available CPU
limits. [NERSC kernel environment instructions](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-set-environment-variables-for-a-python-kernel).

Proceed with cells **1â€“9** to load the notebook helpers, then follow **Workflow and
restart**. Training stays disabled until you explicitly enable cell 13. The full
NERSC installation and reconstruction have not been executed by this documentation
review. [Denoising details](../docs/NAFNET_CT.md) cover architecture and loss settings.


## Workflow and restart

Run cells individually, from the top. Avoid **Run All**, which also executes
reconstruction and inference. Optional cells are labeled in their titles.

| Cells | Purpose |
|---|---|
| 1–4 | Initialize, select paths, inspect scan, configure reconstruction |
| 6, 9–11 | Estimate/freeze COR, enqueue, reconstruct, inspect paired stacks |
| 5, 7–8 | Optional projection alignment and manual COR review |
| 12–13 | Configure training and explicitly enable fresh training/resume |
| 14 | Optional training-history plot, including logs from other runs |
| 15–16 | Inspect denoising and apply it to the full volume |
| 17–20 | Optional queue repair and previewed scratch cleanup |

After a kernel restart, rerun 1–4 for the **same scan and run label**. Cell 6
restores an existing frozen plan; queue reconstruction skips validated completed
stages. Run 11–12 to reconnect existing stacks and saved training configuration.
Cell 13 resumes only if `TrainOutput/resume.pth` exists and you enable training.
Changing reconstruction identity requires a new run label. Updating installed
helper code requires a kernel restart; cell 1 reports its version and source path.

Viewers hold their run context. Selecting a different scan/run or changing
reconstruction settings invalidates old controls; recreate the viewer. Keep
configuration edits together and rerun their configuration cell before continuing.


In [ ]:
# 1. Initialize workflow and check the kernel
from pathlib import Path
import json, os, sys
import krg_workflow
from krg_workflow.runtime import configure_reconstruction_threads, available_cores_minus
THREADS = configure_reconstruction_threads()  # Before scientific imports
from krg_workflow import api as ct, presets, queue, cleanup
from krg_workflow.history import plot_training_history
from krg_workflow.notebook_ui import live_log_box, manual_review
try:
    run.invalidate()
except (NameError, AttributeError):
    pass
run = None
print('Workflow:', krg_workflow.__version__, '| source:', krg_workflow.__file__)
print('Kernel:', sys.executable, '| threads:', THREADS)
print('Slurm:', os.environ.get('SLURM_JOB_ID'), '| GPUs:', os.environ.get('CUDA_VISIBLE_DEVICES'))


## Select paths and a scan — cells 2–3

Set `INPUT_ROOT` and `SCAN_PATH` to your own raw scan. The example is
`$PSCRATCH/raw_microct/sample.h5`; the actual directory must be mounted on the
compute node. Cell 2 lists `.h5` files when the input folder is accessible.
`RUN_LABEL` names this scan's COR/settings/model combination. Use a new label
for changed reconstruction identity or a fresh model comparison.

`ALS_MICROCT_DIR` is the ALS source checkout, and `DENOISE_CMD` is the n2i
executable. `OUTPUT_ROOT_OVERRIDE=None` selects NERSC scratch; set an absolute
`Path` to override it. Cell 3 inspects the selected scan and creates its run
record. These cells do not reconstruct or train. Rerunning scan selection
invalidates old run objects and viewers.


In [ ]:
# 2. Select workspace and scan paths
INPUT_ROOT = Path(os.environ.get('PSCRATCH') or os.environ.get('SCRATCH') or '/alsdata')/'raw_microct'
SCAN_PATH = INPUT_ROOT/'sample.h5'  # Select your raw HDF5 file
RUN_LABEL = 'v3_4_nafnet_1'         # New label for changed source/COR/settings
OUTPUT_ROOT_OVERRIDE = None       # None uses NERSC scratch
ALS_MICROCT_DIR = Path.home()/'microct-n2i-source/microct'
DENOISE_CMD = [str(Path.home()/'microct-n2i-envs/n2i/bin/denoise')]
if run is not None:
    run.invalidate()  # Old viewers must not operate on a new selection
workspace = ct.Workspace.create(ALS_MICROCT_DIR, DENOISE_CMD, OUTPUT_ROOT_OVERRIDE)
print('Selected scan:', SCAN_PATH, '| output root:', workspace.output_root)
if INPUT_ROOT.is_dir():
    print('Available scans:', [p.name for p in sorted(INPUT_ROOT.glob('*.h5'))])


In [ ]:
# 3. Inspect the selected scan
run = ct.inspect_run(workspace, SCAN_PATH, RUN_LABEL, previous=run)
print('Run directory:', run.run_dir)


## Configure reconstruction — cell 4

The notebook calls the ALS backend's `helper.reconstruct` function using the settings below; you do not need to run the upstream ALS notebook separately. `method='fbp'` selects ASTRA FBP (CPU or GPU); `gridrec` selects CPU TomoPy gridrec. The explicit method keeps queued runs consistent across machines. `fc` is the backend's low-pass cutoff; start at 0.75 and compare previews. Reconstruction uses full detector resolution, circular masking, and upstream attenuation scaling.

`preprocessing` operates **after flat/dark normalization, before minus-log**: the outlier threshold is in transmission units, not the legacy raw-count threshold of 750. Zero disables outlier removal. `sm_size=0` disables the upstream SAREPY-style stripe routine; enabling it also requires `snr` and `la_size`. `postprocessing.ringSigma=0` disables Fourier-wavelet stripe removal; a positive value enables it with `ringLevel`. Phase retrieval and exposure-ratio correction are not exposed by this backend wrapper; this notebook assumes equal flat/projection exposure.

Start with corrections off, then compare their effect on previews. Use the same settings for full/even/odd. `als_chunk_sino` limits rows loaded per call; 25 is a conservative starting point. No projection downsampling or 360-to-180 conversion is applied.

| Setting | Interpretation |
|---|---|
| `method` | `fbp`: ASTRA FBP; `gridrec`: TomoPy gridrec. These can produce different filtering and image appearance. |
| `fc` | Cutoff in `(0, 1]`; smaller values smooth more. FBP and gridrec implement filtering differently, so the same number does not imply identical results. |
| `preprocessing.minimum_transmission` | Positive floor on normalized transmission before minus-log; prevents taking log of zero/negative values. |
| `preprocessing.outlier_diff_1D`, `outlier_size_1D` | Transmission difference threshold and window size for filtering along the angle dimension; threshold 0 disables it. |
| `preprocessing.sm_size`, `la_size`, `snr` | Small/large stripe windows and sensitivity for upstream stripe removal; `sm_size=0` disables it. Use suitable odd window sizes when enabled. |
| `postprocessing.ringSigma`, `ringLevel` | Fourier-wavelet damping and decomposition level; `ringSigma=0` disables it. |
| `als_chunk_sino` | Number of detector rows loaded per reconstruction call. It does not limit Noise2Inverse's memory use. |

Inspect changes before freezing the plan. Even a chunk-size or CPU/GPU change changes this notebook's saved settings and requires a new run label. Full/even/odd must be regenerated together when image-forming settings change.

In [ ]:
# 4. Configure reconstruction
RECON_SETTINGS = presets.reconstruction_settings()
# Example: RECON_SETTINGS['als_chunk_sino'] = 25
ALS_USE_GPU = None  # None detects an allocated GPU; False uses CPU
ct.configure_reconstruction(run, workspace, RECON_SETTINGS, use_gpu=ALS_USE_GPU)


## Zoom into image details

Use **Zoom** to enlarge a region and **Center x / Center y** or the arrow buttons
to move around it. **Reset view** returns to the full image. Comparison panels
share the same pixel coordinates, so reconstructed, denoised and residual features
stay aligned. The coordinates and crop size are shown above the images.

**Black / White** adjust display contrast. Matching reconstruction panels share
one contrast window; the residual has its own window. Choose the window with the
Contrast menu. Windows stay fixed when zooming or changing slices; **Reset contrast**
restores the initial full-image window.

Displays use pixel-preserving crops and nearest-neighbor magnification. Zoom does
not change saved reconstructions or run any reconstruction/inference. TIFF browsers
load one row at a time and retain their zoom and contrast when the row changes.

Controls use `ipywidgets`. Without widgets, a static image is shown. Pass
`view_options={'zoom': 8, 'center': (x, y)}` to `ct.inspect_pairs`,
`ct.inspect_alignment`, `ct.inspect_denoising` or `ct.denoise_volume` to select
a static crop. Coordinates are original pixel indices. No interactive Matplotlib
backend is required.


## Optional first/opposite projection check

Skip this section for the usual Vo workflow. The next cell provides a phase-correlation
estimate and an alignment overlay as an independent visual check. It does not supply
Vo's starting point or affect a saved plan.

In [ ]:
# 5. Check projection alignment (optional)
BRIGHT_RATIO = 1.0
alignment_viewer = ct.inspect_alignment(run, bright_ratio=BRIGHT_RATIO)


## Find COR with TomoPy Vo and save the run

Run the next cell, then **enqueue and reconstruct**. It reads three detector-row
sinograms through the same ALS flat/dark normalization, transmission floor, minus-log
and selected preprocessing/postprocessing used for reconstruction. It calls native
`tomopy.find_center_vo()` on each and saves their median midpoint offset when they agree.
No candidate images are reconstructed or scored to estimate COR.

The default rows are at 25%, 50%, and 75% of detector height; choose rows containing
sample if those are empty. Nonfinite/flat rows, estimates near the native Vo limits,
and a row spread greater than 1 pixel require review. The threshold is provisional,
not calibrated accuracy. Approximately uniform 180-degree angles are required.

`VO_OPTIONS` overrides the native parameters shown in `vo_options()`; the default
fine step is 0.1 pixels and `ncore=1` limits native parallelism on supported TomoPy
versions. A fine step is numerical resolution, not measured accuracy. Vo internally
uses its own Fourier coarse/fine procedure; this notebook does not use a reconstruction-image sharpness grid to find COR. Results and logs are saved
under `auto_cor_vo` with the TomoPy version and both detector-center/offset values.

If **REVIEW REQUIRED** appears, change the rows or native bounds, or apply a reviewed
manual COR below. Optional alignment and image previews are available for inspection.
An existing supported run plan restores its saved COR without re-estimation.
Use a **new run label** to obtain a new Vo estimate for a previously frozen scan.

In [ ]:
# 6. Estimate and save COR with TomoPy Vo
AUTO_COR_ROWS = None  # None uses 25%, 50%, 75% of detector height
VO_OPTIONS = {}       # Default ncore=1, step=0.1
# To use allocated CPU cores minus four: VO_OPTIONS = {'ncore': available_cores_minus(4)}
plan, cor_result = ct.estimate_cor(run, rows=AUTO_COR_ROWS, options=VO_OPTIONS)


In [ ]:
# 7. Review COR manually (optional)
RUN_MANUAL_REFINEMENT = False
MANUAL_ROWS = None  # None selects the middle detector row
if RUN_MANUAL_REFINEMENT:
    manual_controls = manual_review(run, rows=MANUAL_ROWS, adjustment=0., apply=False)
else:
    print('Manual review disabled; continue with the saved automatic plan.')


## Optional: save a reviewed COR

Skip this cell when automatic COR has already saved your plan. After manual review,
click **Apply selected COR**, then run the next cell. Alternatively, enter a numeric
midpoint offset in `COR_OVERRIDE` (zero and negative values are valid).

The cell also restores an existing plan after a restart. Frozen reconstruction settings
and COR cannot be changed under the same run label; use a new label for a different COR.
Moving a slider alone does not apply a selection.

In [ ]:
# 8. Save a reviewed COR (optional)
COR_OVERRIDE = None  # Numeric midpoint offset in detector pixels
plan = ct.freeze_selected_cor(run, override=COR_OVERRIDE)


## Queue and reconstruct — cells 9–10

Cell 9 adds the selected `run_plan.json` to
`workspace.output_root/reconstruction_queue_v3.json`. Run it only after COR has produced a
saved plan. Cell 10 prepares even/odd projection views and reconstructs queued
runs **sequentially**; adding an entry alone does not start reconstruction.

The queue can contain several scans/run labels. It records a stable `id` and a
`plan_path` for each entry. Preflight checks each plan and its source/settings;
missing or mismatched entries are blocked, duplicates skipped, and ordinary errors
recorded while later valid runs continue. An interrupt stops further entries.
The queue report is `workspace.output_root/queue_report_v3.json`; each run also has logs and
completion records. On a retry, completed stacks are skipped only after their
TIFFs pass validation; an incomplete tracked stage is reconstructed again.

Cell 11 inspects the run selected in cell 2, not every run that happened to be in
the queue. To remove an entry or repair its path, use cells 17–18. Removing a queue
entry does not delete reconstructed images. The queue lock is
`workspace.output_root/reconstruction_queue_v3.json.lock`; run actions can also use a
`.run.lock` inside their run directory. A lock file marks an active operation;
confirm the owner has stopped before considering stale-lock repair.


In [ ]:
# 9. Add the run to the reconstruction queue
run.assert_current(run.snapshot(), check_source=True)
queued = queue.enqueue_plan(workspace.queue_path, run.plan_path)
print('Queued entries:', len(queued['entries']))


In [ ]:
# 10. Reconstruct queued runs
from krg_workflow.runtime import require_compute_session
require_compute_session()
run.assert_current(run.snapshot(), check_source=True)
report = queue.execute_queue(workspace.queue_path, run.backend, run.settings)
print('Queue report:', report)


## Check the three reconstruction stacks — cell 11

The same row should show the same structures in all three images. Even and odd reconstructions will be noisier and can have more angular undersampling artifacts. Their measured projection indices are disjoint, although shared flat/dark calibration and detector artifacts can leave correlated errors; denoising is not a substitute for this check. Preserve float32 intensity values. The viewer uses the full reconstruction's contrast range for all three images.

Check more than the middle slice, including regions near the top and bottom. Stop before training if boundaries are displaced, the stacks have different orientation, or one view shows a gross intensity mismatch. All output filenames represent the same zero-based detector row: `recon_00000.tiff` is row 0. Shared contrast helps reveal differences but is not a quantitative test of independence or reconstruction accuracy.

In [ ]:
# 11. Inspect reconstruction stacks
pair_viewer = ct.inspect_pairs(run)


## Configure and train

Fresh 2.5D runs default to `nafnet25d`; set `TRAIN_ARCHITECTURE = 'unet'` in
cell 12 for the baseline. **2.5D** uses neighboring axial slices as input and
predicts their center slice. **3D** uses cubic patches and predicts a volume patch;
it uses the 3D U-Net and generally needs more computation/memory. Cell 12 selects
the mode/architecture and applies its preset. Compare five slices first,
then set `TRAIN_OVERRIDES = {'n_slices': 7}` or 9 in a fresh run. BF16 is optional; set `TRAIN_OVERRIDES = {'precision': 'fp32'}` for a precision comparison. These are starting parameters, not tuned optima.

Cell 13 prints a disabled message until you set `RUN_TRAINING = True` yourself.
It then uses all allocated GPUs, retains the live log box, and saves automatic
history plots even on interruption. Patience uses raw evaluation loss, and the
10-epoch history smoothing affects only the plot.

| Setting in cell 12's preset/overrides | Meaning |
|---|---|
| `psz` / `n_slices` | 2.5D square patch width and number of neighboring slices; use an odd slice count |
| `mbsz` | Batch size per GPU; four GPUs with batch 4 process a total batch of 16 |
| `precision` | `bf16` for supported GPUs such as A100; `fp32` for a precision comparison |
| `psz_3d` / `nb_patches_3d` | 3D cube width and number of sampled training patches per epoch |
| `lr` | Adam learning rate; YAML prepared for a new run starts at 0.001 |
| `warmup` | Optimizer updates before adding weighted LCL; initial YAML uses 2000 |
| `maxep` / `patience` | Maximum epochs / consecutive eligible epochs without lower evaluation L1; defaults 2000 / 100 |

An **epoch** visits the sampled training patches and then runs an evaluation pass.
Evaluation currently samples the training dataset again; it is not a separate
held-out accuracy estimate. L1 measures paired absolute error. LCL is an output
contrast regularizer weighted by 0.01 after warmup. L2/MSE and SSIM loss are
recorded only for reference. No learning-rate scheduler is enabled.

Saved training settings and normalization are kept when `TrainOutput` exists;
editing a fresh-run preset does not alter an existing model. Cell 13 automatically
adds `--resume` when a valid `resume.pth` exists, otherwise starts fresh. A U-Net
checkpoint cannot initialize NAFNet. You can run cell 14 on a saved log while
leaving training disabled.


In [ ]:
# 12. Configure training
TRAIN_MODE = '2.5d'             # '2.5d' or '3d'
TRAIN_ARCHITECTURE = 'nafnet25d'  # 'unet' also available; 3D uses unet3d
TRAIN_OVERRIDES = {}           # Example: {'n_slices': 7, 'patience': 100}
training = ct.configure_training(run, workspace, TRAIN_MODE, TRAIN_ARCHITECTURE,
                                 overrides=TRAIN_OVERRIDES, on_line=live_log_box())


In [ ]:
# 13. Train or resume the denoising model
RUN_TRAINING = False  # Set True yourself when ready
ct.train_run(run, workspace, enabled=RUN_TRAINING, on_line=live_log_box())


## Training history

New runs also log L2/MSE and SSIM loss for reference. The automatic plot adds
reference panels when present; the CSV includes them and the fixed SSIM intensity
range. They do not affect optimization, checkpoints or patience. Older logs remain supported.

Training automatically saves and displays the raw epoch-average L1/LCL losses with
a trailing 10-epoch moving average, warmup marker, and best-checkpoint annotations.
The run's `training_history` folder contains PNG, SVG, and extracted CSV files.
It also attempts a partial plot after interruption or failure, when complete epochs exist.

Run the next cell to regenerate a plot for an existing run, or to refresh it from
a growing log. Set `HISTORY_LOG` to another training log if needed. The history cell reads text; it does not load a checkpoint, train a model, or require PyTorch in the ALS kernel. Resumed epochs are
combined, with the latest complete record used when an epoch appears twice.

In a line such as `L1 loss: mean, first => last`, **mean** is the epoch's average
batch value, followed by its first and last batch values. `rate` is seconds for
the complete training/evaluation epoch. L1 and weighted LCL logs sum both paired
directions; new L2/MSE and SSIM-loss logs average them. They report the final GPU
rank's batches, not an aggregate across GPUs.

SSIM loss is `1 - SSIM`; smaller means greater paired similarity. It uses a fixed
normalized-intensity range in `train.ssim_data_range`, inferred once from the two
training stacks if absent and saved in YAML. Compare SSIM only with consistent
range/settings. In 3D, its SSIM/LCL diagnostics use the center axial plane; MSE
uses the entire patch. Metrics against noisy paired targets are not clean-image
accuracy measurements. Reference CSV is also written to
`TrainOutput/results/reference_metrics.csv` and retained on resume.

The live log box shows a scrolling tail while a command runs; the complete log is
saved under the printed `logs/` path. Startup messages may repeat once per GPU
process. ANSI color escapes are stripped from the displayed text. Refreshing
cell 14 reads the current log once; rerun it for another refresh.


In [ ]:
# 14. Show training history (optional)
HISTORY_LOG = None  # Or Path('/absolute/path/to/train.log')
MOVING_AVERAGE_EPOCHS = 10  # Plot smoothing; does not change early stopping
if HISTORY_LOG is None:
    run.assert_current(run.snapshot())
    history_log, history_output = run.run_dir/'logs/train.log', run.run_dir/'training_history'
else:
    history_log = Path(HISTORY_LOG).expanduser().resolve()
    history_output = history_log.parent/'training_history'
history = plot_training_history(history_log, history_output, MOVING_AVERAGE_EPOCHS)


## Inspect denoising — cell 15

**2.5D:** this section generates a single-slice preview. **3D:** the upstream slice command is unavailable; this cell prints next steps until cell 16 has produced a volume, then displays the selected row and residual from that existing volume. Review `TrainOutput/results` before the first 3D volume run. Cell 16 selects `VOLUME_CHECKPOINT` independently. Changing it after a volume exists does not regenerate that volume.

Choose an available checkpoint (`lcl`, `val`, or `edge`) and compare the original, denoised image, and residual at the same row. Look for loss of pores, interfaces, or small structures. The residual should be checked for real features, not judged by smoothness alone.

| `CHECKPOINT` | Checkpoint selection criterion |
|---|---|
| `lcl` | Lowest recorded validation LCL loss. |
| `val` | Lowest recorded validation L1 loss. |
| `edge` | Highest recorded edge score. |

These are internal model-selection scores, not comparisons with a clean reference. A larger edge score can retain noise as well as detail. Compare several rows for removed pores, softened interfaces, and changes in fine structure. Visible anatomy in the residual can indicate removed signal. There is no clean-reference accuracy guarantee from a low loss or a smooth output.

`TEST_SLICE` is the zero-based row index in the full TIFF stack. Rerunning the same test slice writes the same output filename, including when changing checkpoints; preserve a result under another name before comparing saved files later. Cell 16 uses the `CHECKPOINT` selected in cell 15 and the normalization saved in the YAML.

In [ ]:
# 15. Inspect denoising on one slice
CHECKPOINT = 'lcl'  # Compare 'val' or 'edge' where available
TEST_SLICE = run.shape[1]//2
denoising_viewer = ct.inspect_denoising(run, workspace, row=TEST_SLICE,
                                      checkpoint=CHECKPOINT, on_line=live_log_box())


## Denoise the full volume — cell 16

For 2.5D, review several test slices in cell 15 first. For 3D, review the training
preview images, run volume inference, then use cell 15 to inspect the saved rows.
Select an available checkpoint before starting. Training can use several GPUs;
this volume command uses **one GPU**, plus substantial **host RAM**.

This repository's 2.5D profiler uses the actual input channel count; models with
5, 7 or 9 slices are supported. Its prediction buffer is **float32**. For a
624 × 2560 × 2560 volume with 256-pixel patches and 50% overlap, that buffer alone
is about **55 GiB**: `624 × 19 × 19 × 256 × 256 × 4 bytes`. Input, stitching,
temporary arrays and the notebook process require additional RAM. This is a
planning estimate, not a measured peak, and GPU VRAM cannot replace host RAM.
3D stitches predictions incrementally but retains the full input and accumulation
arrays; it still needs substantial host RAM.

Neither the notebook nor this backend deletes an existing volume directory to
restart. Cell 16 refuses to overwrite it. There is **no volume resume**: preserve
partial output and use a new output directory/run for a retry. The CLI's
`--output-dir` can select a separate destination outside this cell, but assembling
independently inferred subvolumes is not automated here. If full 2.5D inference
does not fit, its single-slice command remains available.

NAFNet outputs end in `_nafnet25d`, so they are distinguishable from baseline
outputs. After inference, the notebook checks filenames, dimensions, floating
dtype and finite pixels one slice at a time, then opens a zoomable browser. These
checks add an output-volume read and establish file consistency. Inspect fine
features and residuals before accepting the result.


In [ ]:
# 16. Denoise the full volume
VOLUME_CHECKPOINT = 'lcl'  # This cell has its own explicit checkpoint selection
volume_viewer = ct.denoise_volume(run, workspace, checkpoint=VOLUME_CHECKPOINT,
                                on_line=live_log_box())


## Optional queue inspection and repair — cells 17–18

Run cell 17 with its default empty selections to inspect the queue. The printed
`proposed.entries` contains each entry's `id` and `plan_path`; `preflight` reports
why an entry is valid, blocked or duplicated.

To remove one entry, copy its exact `id` into `REMOVE_ENTRY_IDS`, for example
`REMOVE_ENTRY_IDS = ['copied_entry_id']`, and rerun cell 17. To correct a moved
plan, put `{copied_entry_id: '/absolute/path/to/run_plan.json'}` into
`REPLACE_PLAN_PATHS`. Review the newly printed proposed entries, then set
`APPLY_QUEUE_REPAIR=True` in cell 18 and run it. Repair backs up the exact previous
queue bytes. Entry removal affects the queue only, not run files.

For corrupt queue JSON, set `REBUILD_FROM_PLANS` to an explicit list of valid
absolute plan paths in cell 17. Deduplication and removing confirmed missing
plans are separate opt-in settings; inspect their effects before applying.

If a lock is stale, first confirm its recorded job/kernel has stopped. Inspect
the lock's JSON, copy its token to `STALE_LOCK_TOKEN`, and use the confirmation
flags in cell 18. Never remove the lock of an active reconstruction.


In [ ]:
# 17. Preview queue repair (optional)
REMOVE_ENTRY_IDS = []
REPLACE_PLAN_PATHS = {}  # {stable_entry_id: absolute_plan_path}
DEDUPLICATE_QUEUE = False
REMOVE_CONFIRMED_MISSING = False
REBUILD_FROM_PLANS = None
queue_preview = queue.propose_queue_edit(workspace.queue_path, REMOVE_ENTRY_IDS,
    REPLACE_PLAN_PATHS, DEDUPLICATE_QUEUE, REMOVE_CONFIRMED_MISSING, REBUILD_FROM_PLANS)
print(json.dumps(queue_preview, indent=2))


In [ ]:
# 18. Apply queue repair (optional)
APPLY_QUEUE_REPAIR = False
REMOVE_STALE_QUEUE_LOCK = False
ORIGINAL_JOB_OR_KERNEL_STOPPED = False
STALE_LOCK_TOKEN = None  # Copy from the inspected queue lock
if REMOVE_STALE_QUEUE_LOCK:
    queue.remove_confirmed_stale_lock(queue.queue_lock(workspace.queue_path),
        ORIGINAL_JOB_OR_KERNEL_STOPPED, STALE_LOCK_TOKEN)
if APPLY_QUEUE_REPAIR:
    print('Backup:', queue.apply_queue_edit(workspace.queue_path, queue_preview))


## Optional scratch cleanup — cells 19–20

Use after stopping jobs that use the selected run. Cell 19 is a preview: select
categories in `CLEANUP_CATEGORIES`, rerun it, and inspect every proposed path,
file count, estimated bytes and stated effect. An empty list selects nothing.

| Category | Effect to consider |
|---|---|
| `full`, `even`, `odd` | Removes the corresponding reconstruction stack and invalidates its completion record; queue reconstruction can rebuild it |
| `projection_views` | Removes generated HDF5 view files, which point at the raw scan |
| `denoised` | Removes generated denoising outputs |
| `cor` | Selects generated COR review/preview paths listed in the preview |
| `logs` | Removes run logs; history cannot be regenerated from a deleted log |
| `training` | Removes model/checkpoint outputs and the scan training YAML, including saved normalization; resume/inference then require restoration or fresh training |

For deletion, copy the **exact resolved run directory** from the preview into
`CLEANUP_CONFIRM_RUN` in cell 20, set `CLEANUP_EXECUTE=True` and
`CLEANUP_DRY_RUN=False`, then run that cell. Defaults only preview. Rerun cell 19
after changing any selection. Raw scans, source code, environments, active locks
and symlink/junction paths are protected. Plans and queue entries are retained.

Preserve accepted raw data, TIFFs, model, YAML and plan in your normal durable
storage before choosing cleanup. The report records failures and estimated
reclaimed bytes; it is not a storage backup and free-space updates may lag.


In [ ]:
# 19. Preview scratch cleanup (optional)
CLEANUP_CATEGORIES = []  # cor, projection_views, full, even, odd, denoised, logs, training
run.assert_current(run.snapshot(), check_source=True)
cleanup_plan = ct.read_plan(run.plan_path)
protected_paths = workspace.protected_paths(run)
cleanup_preview = cleanup.cleanup_manifest(cleanup_plan, workspace.output_root,
    CLEANUP_CATEGORIES, protected_paths=protected_paths)
print(json.dumps(cleanup_preview, indent=2))


In [ ]:
# 20. Delete reviewed scratch outputs (optional)
CLEANUP_DRY_RUN = True
CLEANUP_EXECUTE = False
CLEANUP_CONFIRM_RUN = ''  # Exact resolved run directory from the preview
cleanup_report = cleanup.execute_cleanup(cleanup_plan, workspace.output_root, cleanup_preview,
    CLEANUP_DRY_RUN, CLEANUP_CONFIRM_RUN, CLEANUP_EXECUTE, protected_paths=protected_paths)
print('Cleanup:', cleanup_report)
if cleanup_report['status'] != 'dry-run':
    run.invalidate()  # Inspect the run again after deleting generated outputs


## Decisions and recovery

| Situation | What to rerun or inspect |
|---|---|
| Kernel restarted | Run 1–4 with the original scan/run settings, then 6 to restore the plan; run 11–12 to reconnect stacks and training configuration |
| Stale viewer or replaced run | Rerun scan inspection/settings and recreate that viewer |
| COR review required | Choose informative rows/native bounds in 6, or review/apply an offset in 7 and save it in 8; run 5 if an alignment estimate is needed |
| Source/backend/settings differ | Restore the original inputs/settings or use a new run label; do not edit the frozen plan |
| Reconstruction interrupted | Inspect logs/report and rerun 9–10; completed stacks are validated and reused |
| Training interrupted | Preserve YAML/models/logs, run 11–12 with the original model settings, then enable 13 to resume |
| TrainOutput has no resume checkpoint | Preserve it; restore its checkpoint or use a new run label for fresh training |
| Volume inference interrupted | Preserve partial output; there is no automatic volume resume. Resolve the error before retrying with another destination/run |
| Queue needs repair | Preview in 17, apply explicitly in 18 |
| Generated outputs cleaned up | Rerun 3–4 and 6, then 9–12 as needed to rebuild/reconnect outputs |

Read the full log path printed by a failing action. The live box shows only a
tail. GPU utilization alone does not prove an epoch is stalled; check processes,
host RAM and allocation time in the compute terminal. Never remove active locks.

Compare several rows at matched zoom/contrast and inspect residuals for erased
features. Vo row agreement, low loss and a smooth image do not establish accuracy.


## Change notes and validation limits — v3.4

Reusable workflow code now lives in `workflow/src/krg_workflow`. The notebook has
one initialization cell, explicit workspace/run objects and no helper-definition
cells. Raw scan fingerprints, frozen plans, locks, resume checks, allocation
checks and cleanup confirmations are retained. Training still uses L1 plus
weighted LCL; L2/SSIM remain reference metrics. A moving average affects plots
only. The refactor changes no architecture, learning schedule or checkpoint rule.

Local checks cover module imports, Python 3.8 syntax, legacy plan/queue formats,
command generation, history plots, temporary TIFF/HDF5 data and inference. They
do not verify a fresh NERSC installation, real ALS reconstruction or model training.
See [workflow developer guide](../docs/WORKFLOW.md) for the module map and checks.
